# Low-cost Marigold-v2-style depth estimation (free Kaggle GPU)
RGB → frozen SD-VAE encoder → small single-step DiT → depth latent → VAE decoder (stage-2 fine-tuned) → depth.

Training losses: latent MSE, iREPA (DiT features ↔ DINO features of GT depth), and in stage 2 pixel L1 + gradient + Sinkhorn.

**Setup:** Accelerator = GPU (T4 / P100), Internet = ON. Run cells top to bottom. Run the dataset preview and smoke test first.

**Progress:** each training cell prints 4 status lines ([1/4]…[4/4]) and then a live progress bar with losses and s/it. The first batch from NYU takes 1–3 min because it streams from the Hub — that wait is normal.

In [ ]:
!pip install -q -U diffusers transformers datasets accelerate

## 1. Definitions (data, model, losses, helpers, train, inference)

In [ ]:
import sys, argparse, copy, glob, math, os, random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm
from torch.utils.data import Dataset, IterableDataset, DataLoader

SCALE = 0.18215  # SD-VAE latent scale
IMN_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMN_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

### data

In [ ]:
# ----------------------------------------------------------------------------- data
def prep(rgb, depth, size, train):
    """rgb uint8 HxWx3, depth float HxW (any scale; 0 = invalid) -> rgb[-1,1], depth[-1,1], valid mask"""
    rgb = torch.from_numpy(np.ascontiguousarray(rgb)).permute(2, 0, 1).float() / 255
    dep = torch.from_numpy(np.ascontiguousarray(depth)).float()[None]
    _, H, W = rgb.shape
    if train:
        s = random.uniform(0.6, 1.0)
        h, w = int(H * s), int(W * s)
        y, x = random.randint(0, H - h), random.randint(0, W - w)
        rgb, dep = rgb[:, y:y + h, x:x + w], dep[:, y:y + h, x:x + w]
        if random.random() < 0.5:
            rgb, dep = rgb.flip(-1), dep.flip(-1)
    rgb = F.interpolate(rgb[None], (size, size), mode="bilinear", antialias=True, align_corners=False)[0]
    dep = F.interpolate(dep[None], (size, size), mode="nearest")[0]
    valid = dep > 0
    if valid.sum() > 100:  # Marigold-style robust affine normalisation
        lo, hi = torch.quantile(dep[valid], torch.tensor([0.02, 0.98]))
    else:
        lo, hi = dep.min(), dep.max() + 1e-3
    d = ((dep - lo) / (hi - lo + 1e-6)).clamp(0, 1) * 2 - 1
    d = torch.where(valid, d, torch.zeros_like(d))
    return rgb * 2 - 1, d, valid.float()


class HFStream(IterableDataset):
    def __init__(self, name, split, size, train, limit=None):
        self.name, self.split, self.size, self.train, self.limit = name, split, size, train, limit

    def __iter__(self):
        from datasets import load_dataset
        ds = load_dataset(self.name, split=self.split, streaming=True)
        if self.train:
            ds = ds.shuffle(seed=random.randint(0, 10_000), buffer_size=64)
        wi = torch.utils.data.get_worker_info()
        nw, wid = (wi.num_workers, wi.id) if wi else (1, 0)
        n = 0
        for k, ex in enumerate(ds):
            if k % nw != wid:
                continue
            if self.limit and n >= self.limit:
                return
            depth = np.array(ex["depth_map"], dtype=np.float32)
            if depth.ndim == 3:
                depth = depth[..., 0]
            yield prep(np.array(ex["image"].convert("RGB")), depth, self.size, self.train)
            n += 1


class LocalPairs(Dataset):
    def __init__(self, root, size, train):
        self.r = sorted(glob.glob(f"{root}/rgb/*"))
        self.d = sorted(glob.glob(f"{root}/depth/*"))
        assert len(self.r) == len(self.d) > 0, "need matching rgb/ and depth/ files"
        self.size, self.train = size, train

    def __len__(self):
        return len(self.r)

    def __getitem__(self, i):
        from PIL import Image
        rgb = np.array(Image.open(self.r[i]).convert("RGB"))
        p = self.d[i]
        depth = np.load(p).astype(np.float32) if p.endswith(".npy") else np.array(Image.open(p), dtype=np.float32)
        return prep(rgb, depth, self.size, self.train)


class Synthetic(Dataset):  # only for smoke-testing the pipeline
    def __init__(self, size, n=256):
        self.size, self.n = size, n

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        d = F.interpolate(torch.rand(1, 1, 6, 8), (240, 320), mode="bicubic")[0, 0].numpy() + 1.0
        rgb = np.stack([d, d[::-1], d[:, ::-1]], -1)
        rgb = ((rgb - rgb.min()) / (np.ptp(rgb) + 1e-6) * 255).astype(np.uint8)
        return prep(rgb, d.astype(np.float32), self.size, True)


def make_loader(a, split, train):
    if a.data in ("hf", "nyu"):
        ds = HFStream(a.hf_name, split, a.size, train, None if train else a.val_n)
    elif a.data == "local":
        ds = LocalPairs(a.data_dir, a.size, train)
    else:
        ds = Synthetic(a.size)
    shuffle = train and not isinstance(ds, IterableDataset)
    return DataLoader(ds, batch_size=a.bs, shuffle=shuffle, num_workers=a.workers if train else 0,
                      drop_last=train, pin_memory=True, persistent_workers=train and a.workers > 0)


def infinite(dl):
    while True:
        for b in dl:
            yield b

### model

In [ ]:
# ----------------------------------------------------------------------------- model
def sincos_2d(dim, g):
    q = dim // 4
    om = 1.0 / 10000 ** (torch.arange(q).float() / q)
    ys, xs = torch.meshgrid(torch.arange(g).float(), torch.arange(g).float(), indexing="ij")
    f = lambda p: torch.cat([(p.flatten()[:, None] * om).sin(), (p.flatten()[:, None] * om).cos()], 1)
    return torch.cat([f(ys), f(xs)], 1)  # (g*g, dim)


class Block(nn.Module):  # DiT block with adaLN-Zero
    def __init__(self, d, heads):
        super().__init__()
        self.h = heads
        self.n1 = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6)
        self.n2 = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(approximate="tanh"), nn.Linear(4 * d, d))
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(d, 6 * d))
        nn.init.zeros_(self.ada[1].weight); nn.init.zeros_(self.ada[1].bias)

    def forward(self, x, c):
        s1, c1, g1, s2, c2, g2 = self.ada(c)[:, None].chunk(6, -1)
        B, N, D = x.shape
        q, k, v = self.qkv(self.n1(x) * (1 + c1) + s1).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, N, D)
        x = x + g1 * self.proj(a)
        return x + g2 * self.mlp(self.n2(x) * (1 + c2) + s2)


class DiT(nn.Module):
    """Single-step DiT: image latent z_I (4ch) -> depth latent z_d_hat (4ch)."""

    def __init__(self, grid=16, p=2, d=384, depth=12, heads=6, repa_layer=5, repa_dim=384):
        super().__init__()
        self.p, self.g, self.rl = p, grid, repa_layer
        self.embed = nn.Conv2d(4, d, p, p)
        self.register_buffer("pos", sincos_2d(d, grid)[None])
        self.cond = nn.Parameter(torch.zeros(1, d))  # constant "timestep" token (deterministic, 1 step)
        self.cmlp = nn.Sequential(nn.Linear(d, d), nn.SiLU(), nn.Linear(d, d))
        self.blocks = nn.ModuleList(Block(d, heads) for _ in range(depth))
        self.nf = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6)
        self.adaf = nn.Sequential(nn.SiLU(), nn.Linear(d, 2 * d))
        self.out = nn.Linear(d, p * p * 4)
        self.proj = nn.Conv2d(d, repa_dim, 3, padding=1)  # iREPA: conv projector (keeps spatial structure)
        for m in (self.adaf[1], self.out):
            nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, z):
        B = z.shape[0]
        x = self.embed(z).flatten(2).transpose(1, 2) + self.pos
        c = self.cmlp(self.cond).expand(B, -1)
        h = None
        for i, b in enumerate(self.blocks):
            x = b(x, c)
            if i == self.rl:
                h = x
        sh, sc = self.adaf(c)[:, None].chunk(2, -1)
        x = self.out(self.nf(x) * (1 + sc) + sh)
        x = x.view(B, self.g, self.g, self.p, self.p, 4).permute(0, 5, 1, 3, 2, 4)
        return x.reshape(B, 4, self.g * self.p, self.g * self.p), h

    def repa_feats(self, h):
        B, N, D = h.shape
        return self.proj(h.transpose(1, 2).reshape(B, D, self.g, self.g)).flatten(2).transpose(1, 2)

### losses

In [ ]:
# ----------------------------------------------------------------------------- losses
def spatial_norm(f):  # iREPA spatial normalisation over tokens
    return (f - f.mean(1, keepdim=True)) / (f.std(1, keepdim=True) + 1e-6)


def repa_loss(h, f):
    return 1 - F.cosine_similarity(spatial_norm(h.float()), spatial_norm(f.float()), dim=-1).mean()


def pixel_loss(p, g, m):
    return ((p - g).abs() * m).sum() / m.sum().clamp(min=1)


def grad_loss(p, g, m, scales=4):
    tot = 0
    for s in range(scales):
        if s:
            p, g, m = F.avg_pool2d(p, 2), F.avg_pool2d(g, 2), -F.max_pool2d(-m, 2)
        d = (p - g) * m
        tot = tot + (d[..., :, 1:] - d[..., :, :-1]).abs().mean() + (d[..., 1:, :] - d[..., :-1, :]).abs().mean()
    return tot / scales


def sinkhorn_loss(p, g, m, n=256, eps=0.05, iters=30, cw=0.5):
    """Entropic OT between sampled (value, y, x) clouds of prediction and GT.
    Uses detached dual potentials, so the gradient is  sum(P * dC)  (cheap, stable)."""
    B, _, H, W = p.shape
    w = m.flatten(1) + 1e-6
    ip, ig = torch.multinomial(w, n, True), torch.multinomial(w, n, True)

    def cloud(t, idx):
        y, x = (idx // W).float() / H, (idx % W).float() / W
        return torch.stack([t.flatten(1).float().gather(1, idx), cw * y, cw * x], -1)

    C = torch.cdist(cloud(p, ip), cloud(g, ig)) ** 2
    with torch.no_grad():
        Ce, ln = C.detach(), -math.log(n)
        u = torch.zeros(B, n, device=p.device); v = torch.zeros_like(u)
        for _ in range(iters):
            u = -eps * torch.logsumexp((v[:, None, :] - Ce) / eps + ln, 2)
            v = -eps * torch.logsumexp((u[:, :, None] - Ce) / eps + ln, 1)
        P = torch.exp((u[:, :, None] + v[:, None, :] - Ce) / eps + 2 * ln)
    return (P * C).sum((1, 2)).mean()

### helpers

In [ ]:
# ----------------------------------------------------------------------------- helpers
class Ctx:
    """Holds VAE + DiT + (optional) DINO teacher and the encode/decode helpers."""

    def __init__(self, a, need_dino):
        from diffusers import AutoencoderKL
        self.dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.vae = AutoencoderKL.from_pretrained(a.vae).to(self.dev).eval().requires_grad_(False)
        self.grid = a.size // 16
        self.dino = None
        repa_dim = 384
        if need_dino:
            from transformers import AutoModel
            self.dino = AutoModel.from_pretrained(a.dino, torch_dtype=torch.float16).to(self.dev).eval().requires_grad_(False)
            repa_dim = self.dino.config.hidden_size
            self.dsz = self.grid * self.dino.config.patch_size  # -> DINO token grid == DiT token grid
        self.dit = DiT(grid=self.grid, repa_dim=repa_dim).to(self.dev)

    @torch.no_grad()
    def enc(self, x):
        return self.vae.encode(x).latent_dist.mean.float() * SCALE

    def dec(self, z):
        return self.vae.decode(z.float() / SCALE).sample.mean(1, keepdim=True)  # 3ch -> 1ch depth

    @torch.no_grad()
    def dino_feats(self, d):
        x = ((d.repeat(1, 3, 1, 1) + 1) / 2 - IMN_MEAN.to(d.device)) / IMN_STD.to(d.device)
        x = F.interpolate(x, (self.dsz, self.dsz), mode="bilinear", align_corners=False).half()
        return self.dino(pixel_values=x).last_hidden_state[:, -self.grid ** 2:]  # drop CLS/register tokens


def load_ckpt(ctx, path):
    ck = torch.load(path, map_location=ctx.dev)
    ctx.dit.load_state_dict(ck["dit"])
    if "dec" in ck:
        ctx.vae.decoder.load_state_dict(ck["dec"]); ctx.vae.post_quant_conv.load_state_dict(ck["pqc"])


@torch.no_grad()
def evaluate(ctx, ema, dl):
    ema.eval(); mae = rmse = n = 0
    for rgb, d, m in dl:
        rgb, d, m = rgb.to(ctx.dev), d.to(ctx.dev), m.to(ctx.dev)
        with torch.autocast("cuda", dtype=torch.float16):
            zd, _ = ema(ctx.enc(rgb))
            p = ctx.dec(zd).float().clamp(-1, 1)
        e = ((p - d) * m) / 2  # error in [0,1] normalised-depth units
        mae += (e.abs().sum() / m.sum()).item() * len(d); rmse += ((e ** 2).sum() / m.sum()).sqrt().item() * len(d); n += len(d)
    return mae / max(n, 1), rmse / max(n, 1)

### train

In [ ]:
# ----------------------------------------------------------------------------- train
def train(a):
    torch.backends.cuda.matmul.allow_tf32 = True
    print("[1/4] Loading VAE + DINO + DiT (first run downloads weights)...", flush=True)
    ctx = Ctx(a, need_dino=a.w_repa > 0)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0), flush=True)
    else:
        print("WARNING: no GPU found - enable one in Kaggle Settings > Accelerator", flush=True)
    dit, vae = ctx.dit, ctx.vae
    if a.init:
        load_ckpt(ctx, a.init)
    params = [{"params": dit.parameters(), "lr": a.lr}]
    if a.stage == 2:  # "stage 2" VAE decoder is fine-tuned, encoder stays frozen
        dec_p = list(vae.decoder.parameters()) + list(vae.post_quant_conv.parameters())
        for p_ in dec_p:
            p_.requires_grad_(True)
        params.append({"params": dec_p, "lr": a.lr})
        vae.decoder.train()
        try:
            vae.enable_gradient_checkpointing()
        except Exception:
            pass
    opt = torch.optim.AdamW(params, weight_decay=0.01, betas=(0.9, 0.99))
    base = [g["lr"] for g in opt.param_groups]
    scaler = torch.cuda.amp.GradScaler()
    ema = copy.deepcopy(dit).eval().requires_grad_(False)
    print(f"DiT params: {sum(p.numel() for p in dit.parameters()) / 1e6:.1f}M | stage {a.stage}")

    print(f"[2/4] Opening dataset '{a.data}' ...", flush=True)
    it = infinite(make_loader(a, "train", True))
    val_dl = make_loader(a, "validation", False) if a.data in ("hf", "nyu") else None
    print("[3/4] Waiting for first batch (NYU streams from the Hub; can take 1-3 min)...", flush=True)
    t0 = time.time()
    pbar = tqdm(range(1, a.steps + 1), desc=f"stage {a.stage}", dynamic_ncols=True, file=sys.stdout)
    print("[4/4] Training", flush=True)
    for step in pbar:
        lr_f = min(1, step / 500) * 0.5 * (1 + math.cos(math.pi * step / a.steps))
        for g, b in zip(opt.param_groups, base):
            g["lr"] = b * lr_f
        rgb, d, m = [t.to(ctx.dev, non_blocking=True) for t in next(it)]
        if step == 1:
            tqdm.write(f"  first batch ready after {time.time() - t0:.0f}s")
            t0 = time.time()
        d3 = d.repeat(1, 3, 1, 1)
        with torch.autocast("cuda", dtype=torch.float16):
            zI, zd = ctx.enc(rgb), ctx.enc(d3)
            pred, h = dit(zI)
            L = {"lat": F.mse_loss(pred.float(), zd)}
            if a.w_repa > 0:
                L["repa"] = repa_loss(dit.repa_feats(h), ctx.dino_feats(d))
            if a.stage == 2:
                dh = ctx.dec(pred).float()
                L["pix"], L["grad"] = pixel_loss(dh, d, m), grad_loss(dh, d, m)
                L["sink"] = sinkhorn_loss(dh, d, m)
        w = dict(lat=a.w_lat, repa=a.w_repa, pix=a.w_pix, grad=a.w_grad, sink=a.w_sink)
        loss = sum(w[k] * v for k, v in L.items())
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            for pe, p_ in zip(ema.parameters(), dit.parameters()):
                pe.lerp_(p_, 1 - a.ema)
        if step % 10 == 0 or step == 1:
            el = time.time() - t0
            pbar.set_postfix({**{k: round(v.item(), 4) for k, v in L.items()}, "s/it": round(el / max(step - 1, 1), 2) if step > 1 else "-"})
        if step % 100 == 0:
            tqdm.write(f"  step {step}: loss {loss.item():.4f} " + " ".join(f"{k}={v.item():.4f}" for k, v in L.items()))
        stop = bool(a.max_minutes) and (time.time() - t0) / 60 > a.max_minutes
        if step % a.eval_every == 0 or step == a.steps or stop:
            if val_dl is not None:
                tqdm.write("  running validation ...")
                mae, rmse = evaluate(ctx, ema, val_dl)
                tqdm.write(f"  val  MAE {mae:.4f}  RMSE {rmse:.4f}  (normalised [0,1] depth)")
            ck = {"dit": ema.state_dict()}
            if a.stage == 2:
                ck.update(dec=vae.decoder.state_dict(), pqc=vae.post_quant_conv.state_dict())
            torch.save(ck, a.out); tqdm.write(f"  saved {a.out}")
            if stop:
                tqdm.write("  max_minutes reached - stopping."); break

### inference

In [ ]:
# ----------------------------------------------------------------------------- inference
@torch.no_grad()
def predict(a):
    from PIL import Image
    import matplotlib
    ctx = Ctx(a, need_dino=False)
    load_ckpt(ctx, a.init); ctx.dit.eval()
    img = Image.open(a.image).convert("RGB"); W, H = img.size
    x = torch.from_numpy(np.array(img.resize((a.size, a.size), Image.BICUBIC))).permute(2, 0, 1).float()[None] / 127.5 - 1
    with torch.autocast("cuda", dtype=torch.float16):
        x = x.to(ctx.dev)
        zd, _ = ctx.dit(ctx.enc(x)); d = ctx.dec(zd).float()
        # optional test-time flip ensemble
        zf, _ = ctx.dit(ctx.enc(x.flip(-1))); d = (d + ctx.dec(zf).float().flip(-1)) / 2
    d = F.interpolate(d, (H, W), mode="bilinear", align_corners=False)[0, 0].clamp(-1, 1).cpu().numpy()
    d = (d + 1) / 2
    base = os.path.splitext(os.path.basename(a.image))[0]
    np.save(f"/kaggle/working/{base}_depth.npy" if os.path.isdir("/kaggle/working") else f"{base}_depth.npy", d)
    out = f"/kaggle/working/{base}_depth.png" if os.path.isdir("/kaggle/working") else f"{base}_depth.png"
    Image.fromarray((matplotlib.colormaps["Spectral"](d)[..., :3] * 255).astype(np.uint8)).save(out)
    print("saved", out)

## 2. Config

In [ ]:
from argparse import Namespace
cfg = Namespace(
    stage=1, steps=20000, bs=16, lr=None, size=256, ema=0.999, workers=2,
    eval_every=1000, val_n=200, max_minutes=0,
    data="nyu",                        # "nyu" | "local" | "synthetic"
    hf_name="sayakpaul/nyu_depth_v2",  # NYU Depth v2 (47,584 train / 654 val), columns: image, depth_map
    data_dir="",                       # for data="local": DIR/rgb/*, DIR/depth/*
    vae="stabilityai/sd-vae-ft-mse",
    dino="facebook/dinov2-small",      # or facebook/dinov3-vits16-pretrain-lvd1689m (gated)
    init="", out="/kaggle/working/s1.pt",
    w_lat=1.0, w_repa=0.5, w_pix=1.0, w_grad=0.5, w_sink=0.5,
    image="",
)
def run_cfg(**kw):
    a = Namespace(**{**vars(cfg), **kw})
    if a.lr is None: a.lr = 2e-4 if a.stage == 1 else 2e-5
    if a.stage == 1: a.w_pix = a.w_grad = a.w_sink = 0
    return a

### Check GPU + preview NYU Depth v2 (confirms the dataset streams before you start training)

In [ ]:
import torch, matplotlib.pyplot as plt
from datasets import load_dataset
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
ex = next(iter(load_dataset(cfg.hf_name, split="train", streaming=True)))
print("keys:", list(ex.keys()), "| rgb:", ex["image"].size, ex["image"].mode, "| depth:", ex["depth_map"].size, ex["depth_map"].mode)
rgb_t, d_t, m_t = prep(np.array(ex["image"].convert("RGB")), np.array(ex["depth_map"], dtype=np.float32), 256, False)
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow((rgb_t.permute(1, 2, 0) + 1) / 2); ax[0].set_title("RGB"); ax[0].axis("off")
ax[1].imshow(d_t[0], cmap="Spectral"); ax[1].set_title("normalised depth"); ax[1].axis("off"); plt.show()

## 3. Smoke test (~2 min, synthetic data)

In [ ]:
train(run_cfg(data="synthetic", steps=50, bs=8, eval_every=1000, out="/kaggle/working/smoke.pt"))

## 4. Stage 1 — DiT + latent MSE + iREPA (~3–4 h on a T4)

In [ ]:
train(run_cfg(stage=1, steps=20000, bs=16, max_minutes=420, out="/kaggle/working/s1.pt"))

## 5. Stage 2 — fine-tune decoder with pixel / gradient / Sinkhorn losses (~1.5 h)

In [ ]:
train(run_cfg(stage=2, steps=6000, bs=6, max_minutes=240, init="/kaggle/working/s1.pt", out="/kaggle/working/s2.pt"))

## 6. Inference + visualisation

In [ ]:
predict(run_cfg(image="/kaggle/input/YOUR_DATASET/img.jpg", init="/kaggle/working/s2.pt"))
from IPython.display import Image as I, display
display(I("/kaggle/working/img_depth.png"))